# Metric vehicle speed from one camera: demo
RAFT flow + YOLOv8-seg/ByteTrack + ground-plane homography -> km/h per vehicle.
Set `VIDEO`, the real size of the clicked road patch and its 4 pixel points, then run the cells in order. In Colab, first run `!git clone https://github.com/Parul321/cv-vehicle-speed && cd cv-vehicle-speed && pip install -r requirements.txt` and upload your clip.

In [ ]:
import json, cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt

VIDEO = 'clip.mp4'            # fixed-camera clip
WIDTH_M, LENGTH_M = 3.5, 24   # real width and length (metres) of the clicked road patch
# pixel points in the ORIGINAL frame: near-left, near-right, far-right, far-left (example values: replace with yours,
# or run calibrate.py and copy the image_points from calib.json)
IMAGE_POINTS = [[400, 560], [1300, 560], [1100, 250], [500, 250]]

In [ ]:
json.dump({'image_points': IMAGE_POINTS, 'world_points': [[0, 0], [WIDTH_M, 0], [WIDTH_M, LENGTH_M], [0, LENGTH_M]]}, open('calib.json', 'w'))
cap = cv2.VideoCapture(VIDEO); ok, frame = cap.read(); cap.release(); assert ok, 'cannot read video'
cv2.polylines(frame, [np.int32(IMAGE_POINTS)], True, (255, 0, 0), 3)
plt.imshow(frame[:, :, ::-1]); plt.axis('off'); plt.title('Calibration quad (must be a trapezoid on the road)'); plt.show()

## Run the three estimators
RAFT-small (use `raft_large` on a GPU), Farneback baseline, and the tracking-only reference.

In [ ]:
!python speed_estimator.py $VIDEO --flow raft_small --width 640 --out demo_raft
!python speed_estimator.py $VIDEO --flow farneback --width 640 --out demo_farn
!python speed_estimator.py $VIDEO --flow track --width 640 --out demo_track

In [ ]:
r = pd.read_csv('demo_raft.csv')
print(r.speed_kmh.describe().round(1))
r.groupby('track_id').speed_kmh.median().plot.hist(bins=15)
plt.xlabel('median speed per vehicle (km/h)'); plt.show()

## Compare
With `gt.csv` (`track_id,gt_kmh`) the table is accuracy against ground truth; with `demo_track.csv` as reference it is agreement with the tracking-only method (not ground truth).

In [ ]:
!python compare.py demo_track.csv demo_raft.csv demo_farn.csv
from IPython.display import Image, display
display(Image('compare_plot.png'))